In [8]:
import json
import krippendorff
import numpy as np

In [3]:
with open('final_evaluated_cases_1.json', 'r', encoding='utf-8') as f:
    eval1 = json.load(f)

with open('final_evaluated_cases_2.json', 'r', encoding='utf-8') as f:
    eval2 = json.load(f)

In [4]:
eval1[0]

{'CNR': 'KLHC010074352013',
 'case_details': 'Applicant applied for regular-bail. The age of the applicant/s is/are [27]. The health condition of the applicant is none. Past criminal records for the applicant do not exist. The relevant statutes are: 376 IPC, 156 CrPC, 34 IPC. The applicant is in custody for 15 days. The petitioner is the first accused in Crime No. 1290 of 2013 of Kadavanthra Police Station, who is alleged to have committed offences punishable under Section 376 read with Section 34 of Indian Penal Code. The allegation is that the petitioner herein, who is the first accused, was staying with the second accused who had let him to occupy a hostel run by her. The allegation is that the first accused with the connivance of the second accused began to induce the victim to develop affinity towards the first accused. Though the victim was reluctant to do so and she had expressed her disinclination for that course, the second accused pursued her attempt. It is alleged that when 

In [5]:
ft_fa = [[item['evaluation']['ft_evaluation']['fa'] for item in eval1], [item['evaluation']['ft_evaluation']['fa'] for item in eval2]]
ft_c1 = [[item['evaluation']['ft_evaluation']['c1'] for item in eval1], [item['evaluation']['ft_evaluation']['c1'] for item in eval2]]
ft_c2 = [[item['evaluation']['ft_evaluation']['c2'] for item in eval1], [item['evaluation']['ft_evaluation']['c2'] for item in eval2]]

base_fa = [[item['evaluation']['baseline_evaluation']['fa'] for item in eval1], [item['evaluation']['baseline_evaluation']['fa'] for item in eval2]]
base_c1 = [[item['evaluation']['baseline_evaluation']['c1'] for item in eval1], [item['evaluation']['baseline_evaluation']['c1'] for item in eval2]]
base_c2 = [[item['evaluation']['baseline_evaluation']['c2'] for item in eval1], [item['evaluation']['baseline_evaluation']['c2'] for item in eval2]]

In [14]:
def krippendorff_alpha(data):
    data_array = np.array(data)
    alpha = krippendorff.alpha(reliability_data=data_array, level_of_measurement='ordinal')
    return alpha

In [15]:
alpha_ft_fa = krippendorff_alpha(ft_fa)
alpha_ft_c1 = krippendorff_alpha(ft_c1)
alpha_ft_c2 = krippendorff_alpha(ft_c2)
alpha_base_fa = krippendorff_alpha(base_fa)
alpha_base_c1 = krippendorff_alpha(base_c1)
alpha_base_c2 = krippendorff_alpha(base_c2)

In [16]:
print("alpha_ft_fa:", alpha_ft_fa)
print("alpha_ft_c1:", alpha_ft_c1)
print("alpha_ft_c2:", alpha_ft_c2)
print("\n")
print("alpha_base_fa:", alpha_base_fa)
print("alpha_base_c1:", alpha_base_c1)
print("alpha_base_c2:", alpha_base_c2)

alpha_ft_fa: 0.19935508217136155
alpha_ft_c1: 0.3626098807495741
alpha_ft_c2: 0.2689362305684785


alpha_base_fa: 0.6977033413020515
alpha_base_c1: 0.8269542477210209
alpha_base_c2: 0.7019479218828242


In [19]:
with open('evaluate_this_augmented.json', 'r', encoding='utf-8') as f:
    re_eval = json.load(f)

In [20]:
len(re_eval)

50

In [22]:
pred = {}
for item in re_eval:
    pred[item['CNR']] = {
        'pred_ft': item['pred_ft'],
        'pred_base': item['pred_baseline']
    }

In [26]:
eval1.sort(key=lambda x: x['CNR'])
eval2.sort(key=lambda x: x['CNR'])

In [30]:
eval1[0]

{'CNR': 'CGHC010094062018',
 'case_details': 'Applicant applied for anticipatory-bail. The age of the applicant/s is/are [39, 38]. The health condition of the applicant is none. Past criminal records for the applicant do not exist. The relevant statutes are: 186 IPC, 353 IPC, 294 IPC, 506 IPC, 34 IPC. The applicants are accused in connection with crime No.638/2017, registered at Police Station â\x80\x93 Mahasamund, District- Mahasamund (Chhattisgarh) for the offences punishable under Sections 294, 506, 353, 186, 34(1) read with Section 34 of the Indian Penal Code. The factual incident is that the complainant - (Vaibhav Shukla, Traffic Sub-inspector) was demanding illegal gratification from the trucks that were transporting stones because of which the dispute has taken place. The applicants have also filed a complaint on the same day against the present complainant due to which the FIR has been lodged against them.\nArguments supporting the bail application are It is submitted by learne

In [31]:
final_eval = []
for e1, e2 in zip(eval1, eval2):
    assert e1['CNR'] == e2['CNR'], "CNR mismatch between eval1 and eval2"
    e1['pred_ft'] = pred[e1['CNR']]['pred_ft']
    e1['pred_base'] = pred[e1['CNR']]['pred_base']
    e2['pred_ft'] = pred[e2['CNR']]['pred_ft']
    e2['pred_base'] = pred[e2['CNR']]['pred_base']
    eval = {
        'CNR': e1['CNR'],
        'case_details': e1['case_details'],
        'outcome': e1['outcome'],
        'reason_court': e1['reason_court'],
        'pred_base': e1['pred_base'],
        'pred_ft': e1['pred_ft'],
        'reason_baseline': e1['reason_baseline'],
        'reason_ft': e1['reason_ft'],
        'evaluation': {
            'evaluator_1': e1['evaluation'],
            'evaluator_2': e2['evaluation']
        }
    }
    final_eval.append(eval)

In [32]:
with open('final_evaluated_cases_1.json', 'w', encoding='utf-8') as f:
    json.dump(eval1, f, indent=4)
with open('final_evaluated_cases_2.json', 'w', encoding='utf-8') as f:
    json.dump(eval2, f, indent=4)
with open('merged_human_evaluation.json', 'w', encoding='utf-8') as f:
    json.dump(final_eval, f, indent=4)

In [35]:
import json
import pandas as pd

# load json
with open("merged_human_evaluation.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# flatten nested json
df = pd.json_normalize(data)

# save csv
df.to_csv("output.csv", index=False)

print("CSV saved as output.csv")

CSV saved as output.csv
